In [1]:
import torch
import torch.optim
from dataloader import SelfDataSet
from log import Logger
from plot import plot_picture
import os
import torch.nn as nn
from unet import UNet
from torch.utils.data import DataLoader
from torch import optim
import time
import matplotlib.pyplot as plot
from torch.cuda.amp import autocast, GradScaler  # 导入混合精度工具


# 定义训练函数
def Train_Unet(net, device, data_path, batch_size=(), epochs=(), lr=()):
    # 加载数据集（无数据增强）
    train_dataset = SelfDataSet(data_path)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # 定义优化算法
    opt = optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)

    # 使用 CrossEntropyLoss
    loss_fun = nn.CrossEntropyLoss()

    # 创建日志记录
    Unet_train_txt = Logger('Unet_train.txt')

    # 初始化梯度缩放器
    scaler = GradScaler()

    # 学习率调度器
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    # 最优模型的初始损失
    best_loss = float('inf')

    for epoch in range(epochs):
        net.train()
        running_loss = 0.0
        i = 0
        start_time = time.perf_counter()

        for image, label in train_loader:
            opt.zero_grad()
            image = image.to(device=device, dtype=torch.float32)
            label = label.to(device=device, dtype=torch.long)  # CrossEntropyLoss 需要 long 类型标签

            # 去掉标签的通道维度，调整为 (batch_size, height, width)
            label = label.squeeze(1)  # 去掉多余的通道维度

            # 使用 autocast 进行混合精度计算
            with autocast():
                pred = net(image)  # 网络输出 logits
                loss = loss_fun(pred, label)

            # 使用梯度缩放进行反向传播
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=1.0)  # 梯度裁剪
            scaler.step(opt)
            scaler.update()

            i += 1
            running_loss += loss.item()

        # 计算每轮的平均损失
        loss_avg_epoch = running_loss / i
        Unet_train_txt.write(f"{loss_avg_epoch:.4f}\n")
        end_time = time.perf_counter()

        # 调整学习率
        scheduler.step()

        # 打印训练信息
        print(f"Epoch {epoch + 1}/{epochs}, Avg Loss: {loss_avg_epoch:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}, Time: {end_time - start_time:.2f}s")

        # 保存最优模型
        if loss_avg_epoch < best_loss:
            best_loss = loss_avg_epoch
            state = {'net': net.state_dict(), 'opt': opt.state_dict(), 'epoch': epoch}
            torch.save(state, 'model_best.pth')
            print(f"Epoch {epoch + 1}: Best model saved with loss {best_loss:.4f}")

    torch.cuda.empty_cache()
    Unet_train_txt.close()


def tryGPU(i=0):
    if torch.cuda.device_count() >= i + 1:
        return torch.device(f'cuda:{i}')
    return torch.device('cpu')


# 主函数
if __name__ == '__main__':
    torch.backends.cudnn.benchmark = True  # 提高卷积计算速度
    device = tryGPU()
    print(device)
    net = UNet(1, 2, bilinear=False)  # 修改为 2 个输出通道（适用于二分类任务）
    net.to(device=device)

    # 数据集路径
    data_path = "./train_image/"
    Train_Unet(net, device, data_path, epochs=20, batch_size=32, lr=0.001)

    # 绘制训练曲线
    plot_picture('Unet_train.txt')

cpu


/dssg/share/conda_env/ms8410/lib/python3.11/site-packages/torch/cuda/amp/grad_scaler.py:120: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn("torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.")
/dssg/share/conda_env/ms8410/lib/python3.11/site-packages/torch/amp/autocast_mode.py:204: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn('User provided device_type of \'cuda\', but CUDA is not available. Disabling')


Epoch 1/20, Avg Loss: 0.7385, LR: 0.000994, Time: 46.60s
Epoch 1: Best model saved with loss 0.7385
Epoch 2/20, Avg Loss: 0.4303, LR: 0.000976, Time: 42.13s
Epoch 2: Best model saved with loss 0.4303
Epoch 3/20, Avg Loss: 0.3327, LR: 0.000946, Time: 42.04s
Epoch 3: Best model saved with loss 0.3327
Epoch 4/20, Avg Loss: 0.2785, LR: 0.000905, Time: 42.10s
Epoch 4: Best model saved with loss 0.2785
Epoch 5/20, Avg Loss: 0.2287, LR: 0.000854, Time: 42.07s
Epoch 5: Best model saved with loss 0.2287
Epoch 6/20, Avg Loss: 0.1927, LR: 0.000794, Time: 42.06s
Epoch 6: Best model saved with loss 0.1927
Epoch 7/20, Avg Loss: 0.1634, LR: 0.000727, Time: 42.07s
Epoch 7: Best model saved with loss 0.1634
Epoch 8/20, Avg Loss: 0.1397, LR: 0.000655, Time: 42.17s
Epoch 8: Best model saved with loss 0.1397
Epoch 9/20, Avg Loss: 0.1227, LR: 0.000578, Time: 42.05s
Epoch 9: Best model saved with loss 0.1227
Epoch 10/20, Avg Loss: 0.1092, LR: 0.000500, Time: 42.19s
Epoch 10: Best model saved with loss 0.109

FileNotFoundError: [Errno 2] No such file or directory: './train_image/43.png'